# Bibliotecas

In [ ]:
# Sempre extrair e salvar planinha de IDs

In [ ]:
# alterar !

# mudar no pacote de notas para aqueles que não tiveram indicador puxar da ondem de clientes e de dj e rl 
# tirar notas do plano de PLANOPODA6M = pagina croqui
#  priorizar circuitos de 34,5 



In [ ]:
import os
import pandas as pd
import numpy as np

import subprocess
import shutil
import requests
import zipfile
import time
import psutil
from selenium import webdriver
from selenium.webdriver.edge.service import Service as EdgeService
from selenium.webdriver.edge.options import Options as EdgeOptions
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import WebDriverException


from datetime import datetime, timedelta

In [ ]:
# Obtém o nome do usuário do sistema
usuario = os.getlogin()
print(usuario)

In [ ]:
#caminhos
#caminho_downloads = f'C:\\Users\\{usuario}\\Downloads\\'
caminho_base = f'C:\\Users\\{usuario}\\EDP\\O365_Planejamento_Manutencao_EDP_Brasil - Documentos\\PLANO MANUTENÇÃO\\Pacote de notas\\SP\\Base para Geração de Pacotes\\'
caminho_aux = f'C:\\Users\\{usuario}\\EDP\\O365_Planejamento_Manutencao_EDP_Brasil - Documentos\\PLANO MANUTENÇÃO\\Pacote de notas\\SP\\Auxiliares\\'
# email_envio = 'danilop.vilela@edp.com'

# Login_SAP = '709496'
# Senha_SAP = '@Edp2025'

In [ ]:
#ano atual
ano = datetime.now().year
# Data de hoje
hoje = pd.Timestamp.today().normalize()
print(ano,hoje)

# Baixa base de ocorrencias

In [ ]:
arquivo1 = os.path.join(caminho_base, "Base_Ocorrencias_SP.csv")
arquivo2 = os.path.join(caminho_base, "Base_Indicadores_Movel_SP.csv")

# Remove se já existir
if os.path.exists(arquivo1):
    os.remove(arquivo1)
else:
    pass

# Remove se já existir
if os.path.exists(arquivo2):
    os.remove(arquivo2)
else:
    pass

In [ ]:
#Sempre instala edgedriver da versão atual
caminho_downloads = caminho_base

for proc in psutil.process_iter(['name']):
    if proc.info['name'] == 'msedgedriver.exe':
        try:
            proc.kill()
        except Exception:
            pass
def acessar():
    """Acessa o site do Coffee"""
    global navegador

    # Determina a versão do Microsoft Edge instalada
    try:
        result = subprocess.run(["reg", "query", "HKEY_CURRENT_USER\\Software\\Microsoft\\Edge\\BLBeacon", "/v", "version"], capture_output=True, text=True, shell=True)
        edge_version = None
        for line in result.stdout.splitlines():
            if "version" in line:
                edge_version = line.split()[-1]
                break
        if not edge_version:
            raise Exception("Não foi possível determinar a versão do Microsoft Edge.")
    except Exception as e:
        print(f"Erro ao obter a versão do Microsoft Edge: {e}")
        os._exit(1)

    # Monta o link de download do EdgeDriver
    edgedriver_url = f"https://msedgedriver.microsoft.com/{edge_version}/edgedriver_win64.zip"

    # Caminho para salvar o arquivo baixado e descompactar
    download_path = "C:\\Selenium\\edgedriver_win64.zip"
    extract_path = "C:\\Selenium\\"

    # Cria a pasta C:\Selenium se ela não existir
    if not os.path.exists(extract_path):
        os.makedirs(extract_path)

    # Faz o download do EdgeDriver
    try:
        response = requests.get(edgedriver_url, stream=True, verify=False)
        if response.status_code == 200:
            with open(download_path, "wb") as file:
                shutil.copyfileobj(response.raw, file)
        else:
            raise Exception(f"Falha no download do EdgeDriver. Status code: {response.status_code}")
    except Exception as e:
        print(f"Erro ao baixar o EdgeDriver: {e}")
        os._exit(1)

    # Descompacta o arquivo baixado
    try:
        with zipfile.ZipFile(download_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        os.remove(download_path)  # Remove o arquivo zip após descompactar
    except Exception as e:
        print(f"Erro ao descompactar o EdgeDriver: {e}")
        os._exit(1)

    # Configurando opções do navegador
    opt = EdgeOptions()
    opt.add_argument("--start-maximized") #Abre em tela cheia
    #opt.add_argument("--headless")  # Executa o navegador em modo invisível

    prefs = {
        "download.default_directory": caminho_downloads,   # pasta para downloads
        "download.prompt_for_download": False,        # não perguntar
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True                  # evita bloqueio
    }
    opt.add_experimental_option("prefs", prefs)

    # Configurando o serviço com o caminho do executável msedgedriver.exe
    service = EdgeService("C:\\Selenium\\msedgedriver.exe")

    navegador = webdriver.Edge(service=service, options=opt)

    #link do site que deseja abrir
    navegador.get("https://adb-2199566635981619.19.azuredatabricks.net/ajax-api/2.0/fs/files/Volumes/poseidon_uc/group_uc/ddpm/Danilo/Base_Ocorrencias_SP.csv")
    return navegador
acessar()
time.sleep(1)

In [ ]:
#Clica para entrar no databricks, faz o Download automatico do arquivo
navegador.find_element(By.XPATH, '//*[@id="login-page"]/div/div/div[3]/a/span').click()

In [ ]:
time.sleep(5)
navegador.get("https://adb-2199566635981619.19.azuredatabricks.net/ajax-api/2.0/fs/files/Volumes/poseidon_uc/group_uc/ddpm/Danilo/Base_Indicadores_Movel_SP.csv")
time.sleep(5)

In [ ]:
#fecha navegador
count = 0
while (os.path.exists(arquivo1) == False) and (os.path.exists(arquivo2) == False) (count <=300) :
    time.sleep(10)
    count = count + 10 

navegador.quit()

In [ ]:
Indicadores = pd.read_csv(
    fr'{caminho_base}Base_Indicadores_Movel_SP.csv',
    sep=";",   # tenta ; em vez de ,
    encoding="utf-8"  # se der erro de acento, usar "latin1"
)

display(Indicadores)

In [ ]:
Ocorrencias = pd.read_csv(
    fr'{caminho_base}Base_Ocorrencias_SP.csv',
    sep=";",   # tenta ; em vez de ,
    encoding="utf-8"  # se der erro de acento, usar "latin1"
)

display(Ocorrencias)

## Cria dinamica por blocos

In [ ]:
colunas_numericas = ['CI', 'CHI']
for col in colunas_numericas:
    Ocorrencias[col] = (
        Ocorrencias[col]
            .astype(str)
            .str.replace(',', '.', regex=False)
            .str.strip()
    )

    Ocorrencias[col] = pd.to_numeric(
        Ocorrencias[col],
        errors='coerce'
    ).fillna(0)

# =========================
# Definir tipos finais
# =========================

Ocorrencias['CI'] = Ocorrencias['CI'].round().astype('Int64')
Ocorrencias['CHI'] = Ocorrencias['CHI'].astype(float)

Ocorrencias

In [ ]:
causas_gerenciaveis = (11, 20, 130, 180, 181, 182, 200, 322, 323, 324, 327, 329, 491, 492, 820, 880, 890)

Ocorrencias = Ocorrencias[Ocorrencias['TIPO_DESLIGAMENTO'] == 'ACIDENTAL']
print(len(Ocorrencias))
Ocorrencias = Ocorrencias[Ocorrencias['CAUSA'].isin(causas_gerenciaveis)]

print(len(Ocorrencias))

In [ ]:
# Cria coluna de data a partir de ANO, MES, DIA
Ocorrencias['DATA'] = pd.to_datetime(Ocorrencias[['ANO', 'MES', 'DIA']].astype(str).agg('-'.join, axis=1), format='%Y-%m-%d')

# Data limite: 90 dias atrás da data atual
data_limite = datetime.now() - timedelta(days=90)

# Filtra por datas nos últimos 90 dias
df_ultimos_90 = Ocorrencias[Ocorrencias['DATA'] >= data_limite]

# Conta distintos ORDEM_ID nos últimos 90 dias por EQ_INTERRUPCAO
qtd_90d = df_ultimos_90.groupby('EQ_INTERRUPCAO')['ORDEM_ID'].nunique().reset_index().rename(columns={'ORDEM_ID': 'QTD_Ocorrencias_90d'})

# Junta com o grouped_df original
grouped_df = Ocorrencias.groupby('EQ_INTERRUPCAO').agg({'ORDEM_ID': 'nunique','CI': 'sum', 'CHI': 'sum'}).reset_index().rename(columns={'ORDEM_ID': 'QTD_Ocorrencias'})
grouped_df = grouped_df.merge(qtd_90d, on='EQ_INTERRUPCAO', how='left').fillna({'QTD_Ocorrencias_90d': 0})

# # =========================
# # 6️⃣ Garantir que tudo vire número
# # =========================

colunas_numericas = ['QTD_Ocorrencias', 'QTD_Ocorrencias_90d', 'CI', 'CHI']

for col in colunas_numericas:
    grouped_df[col] = (
        grouped_df[col]
            .astype(str)
            .str.replace(',', '.', regex=False)
            .str.strip()
    )

    grouped_df[col] = pd.to_numeric(
        grouped_df[col],
        errors='coerce'
    ).fillna(0)

# =========================
# 7️⃣ Definir tipos finais
# =========================

grouped_df['QTD_Ocorrencias'] = grouped_df['QTD_Ocorrencias'].astype('Int64')
grouped_df['QTD_Ocorrencias_90d'] = grouped_df['QTD_Ocorrencias_90d'].astype('Int64')
grouped_df['CI'] = grouped_df['CI'].round().astype('Int64')
grouped_df['CHI'] = grouped_df['CHI'].astype(float)

# =========================
# 8️⃣ Ajuste final na chave
# =========================
grouped_df['EQ_INTERRUPCAO'] = grouped_df['EQ_INTERRUPCAO'].astype(str) + '9'

Dim_Ocorrencias = grouped_df

display(Dim_Ocorrencias)

# Versão de indicadores paleativa

In [ ]:
Dim_Ocorrencias = pd.read_csv(
    fr'{caminho_base}Ocorrencias_indicadores_sp.csv',
    sep=";",   # tenta ; em vez de ,
    encoding="utf-8"  # se der erro de acento, usar "latin1"
)

Dim_Ocorrencias['EQ_INTERRUPCAO'] = Dim_Ocorrencias['EQ_INTERRUPCAO'].astype(str) + '9'

# Colunas inteiras
colunas_int = ['QTD_Ocorrencias', 'QTD_Ocorrencias_90d', 'CI']

# Colunas decimais
colunas_float = ['CHI', 'DEC_EMPRESA', 'FEC_EMPRESA']

Dim_Ocorrencias[colunas_float] = (
    Dim_Ocorrencias[colunas_float]
    .replace(',', '.', regex=True)
    .apply(pd.to_numeric, errors='coerce')
)
# Conversão
Dim_Ocorrencias[colunas_int] = (
    Dim_Ocorrencias[colunas_int]
    .apply(pd.to_numeric, errors='coerce')
    .astype('Int64')  # inteiro que aceita NaN
)

Dim_Ocorrencias[colunas_float] = (
    Dim_Ocorrencias[colunas_float]
    .apply(pd.to_numeric, errors='coerce')
    .astype(float)
)
display(Dim_Ocorrencias)

In [ ]:
Dim_Ocorrencias.columns

# Base de notas

## Base de notas do SQL (Não tem supervisão)

In [ ]:
# import pandas as pd
# import pyodbc
# from sqlalchemy import create_engine

# # Lê o arquivo de Acesso ao SQL Server
# df_acessoSQL = pd.read_csv('\\\\fscoc10\\dep\\DDPM\\COFFEE\\Gerador de Notas\\AcessoSQL.csv', sep=';')
# u_sql = df_acessoSQL['Acesso'][0]
# p_sql = df_acessoSQL['Acesso'][1]

# ## SQL Server
# # Configuração da string de conexão
# print('Criando string de conexão...')
# server = 'edpbredb.database.windows.net'
# database = 'ddpm'
# username_ad = u_sql  # Substitua pelo seu usuário
# password = p_sql  # Substitua pela sua senha

# # String de conexão
# conn_str = (
#     f'DRIVER={{ODBC Driver 17 for SQL Server}};'
#     f'SERVER={server};'
#     f'DATABASE={database};'
#     f'Authentication=ActiveDirectoryPassword;'
#     f'UID={username_ad};'
#     f'PWD={password};'
#     f'TrustServerCertificate=yes;'
# )

# # Conexão com o banco de dados
# print('Conectando ao banco de dados...')
# connection_url = f"mssql+pyodbc:///?odbc_connect={conn_str}"
# engine = create_engine(connection_url)

# ## Ler DataFrame do SQL Server
# print('Lendo DataFrame do SQL Server...')

# #df = pd.read_sql('SELECT * from NotasDONR', engine)

# # Imprimir DataFrame
# #print(df)

In [ ]:
# notas = pd.read_sql('''SELECT *  FROM NotasDONR
#                  WHERE
#                      EMPRESA = 380 --391 SP-- Mudar filtro da empresa 
#                      AND (local_instalacao IS NULL or local_instalacao LIKE '7%') -- Mudar filtro de local de instalação 
#                      AND [descrição_conjunto] IN (
#                         'BF-CHAVE FUSIVEL <34,5 kV- OPEX',
#                         'MANUT.  EQUIPAMENTOS - OPEX',
#                         'MANUT. CIRC. PRIMARIO - MT - OPEX',
#                         'MANUT. CIRC. SECUNDARIO-BT - OPEX',
#                         'PODA DE ARVORES - OPEX'
#                      )
#                      AND id_sap NOT IN ('99999999', '999999999')
#                      AND Status_SAP <> 'Encerrado' AND Status_SAP <> 'Cancelado'
#                      AND recebido >= '2024-01-01T00:00:00'

#                  ''', engine)

# # Imprimir DataFrame
# display(notas)

In [ ]:
# notas['recebido'] = pd.to_datetime(notas['recebido'], errors='coerce')
# # Cria colunas 'ANO', 'MES' e 'DIA' a partir da coluna 'Data'
# notas['ANO'] = notas['recebido'].dt.year
# notas['MES'] = notas['recebido'].dt.month
# notas['DIA'] = notas['recebido'].dt.day

In [ ]:
# display(notas['ANO'].unique())

## Le base de Notas

In [ ]:
#Baixar tabelas de ids pendentes 

In [ ]:
Notas = pd.read_excel(
    fr'{caminho_base}Notas_Pendentes.xlsx')

#display(Notas)

In [ ]:
#Garante que todas as notas estejam pedentes e sejam de manutenção ou poda
Notas = Notas[Notas['descrição_conjunto'].str.contains('MANUT.|PODA', case=False, na=False)]
Notas = Notas[Notas['Data_encerramento_exec'].isnull()]

In [ ]:
# Criando coluna com diferença em dias
Notas['tempo de nota'] = (hoje - Notas['recebido']).dt.days
display(Notas)

In [ ]:
condicoes = [
    Notas['tempo de nota'] <= 365,
    (Notas['tempo de nota'] > 365) & (Notas['tempo de nota'] <= 730),
    Notas['tempo de nota'] > 730
]

categorias = [
    'Até 1 ano',
    'Até 2 anos',
    'Mais de 2 anos'
]

Notas['Classificação Tempo'] = np.select(condicoes, categorias, default='Sem data')
Notas = Notas[Notas['Classificação Tempo'].isin(['Até 1 ano','Até 2 anos'])]

In [ ]:
Notas_bk = Notas
#Notas = Notas_bk

In [ ]:
len(Notas)

## Remove Medidas que não são manutenção

In [ ]:
Notas = Notas[~Notas['componente_novo'].str.contains('abelha|Aprumar poste|Separador|Placa', case=False, na=False)]

In [ ]:
Notas

# Le Hierarquia

In [ ]:
#tabela é  tabela hierarquia do sql ver posibilidade de pegar do sql 

In [ ]:
hierarquia = pd.read_excel(
    fr'{caminho_aux}Hierarquia_SP_2025_Supervisão.xlsx',
)

display(hierarquia)

# Le capacidade de execução

In [ ]:
capacidade = pd.read_excel(
    fr'{caminho_aux}Capacidade Execução.xlsx', #Precisa ser atulizado
    sheet_name='Capacidade_Total'
)

display(capacidade)

In [ ]:
Divisão_Supervisão = pd.read_excel(
    fr'{caminho_aux}Capacidade Execução.xlsx', #Precisa ser atulizado
    sheet_name='Divisão por Supervisão'
)

display(Divisão_Supervisão)

In [ ]:
# print("⚠️ Aviso: Na celula abaixo selecione os meses correspondetes para novo pacote. Pressione Enter para continuar...")
# input()


In [ ]:
import ipywidgets as widgets
from IPython.display import display

meses = [
    "Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
    "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro"
]

titulo = widgets.HTML("<b>Selecione os meses do novo pacote:</b>")
seletor_meses = widgets.SelectMultiple(options=meses, value=(), description='Meses', rows=12)
btn_confirmar = widgets.Button(description="Salvar Meses", button_style="success")
saida = widgets.Output()

meses_selecionados = []

def confirmar_meses(_):
    global meses_selecionados
    meses_selecionados = list(seletor_meses.value)
    with saida:
        saida.clear_output()
        print("Meses selecionados:", meses_selecionados)

btn_confirmar.on_click(confirmar_meses)

display(titulo, seletor_meses, btn_confirmar, saida)


In [ ]:
meses_selecionados = list(seletor_meses.value)

if not meses_selecionados:
    raise ValueError("Selecione pelo menos 1 mês antes de continuar.")

print("Meses selecionados:", meses_selecionados)


In [ ]:
capacidade = capacidade[capacidade['Atributo'].isin(meses_selecionados)]

In [ ]:
capacidade_total = capacidade.groupby('CSD').agg({'Valor': 'sum'}).reset_index()
capacidade_total['META CSD'] = capacidade_total['Valor'] * 0.8
display(capacidade_total)

In [ ]:
Divisão_Supervisão_join = Divisão_Supervisão.merge(capacidade_total, left_on='CSD', right_on='CSD', how='left').drop('Valor', axis=1)

Divisão_Supervisão_join['QTD Supervisão'] = np.ceil(
    Divisão_Supervisão_join['META CSD'] *
    Divisão_Supervisão_join['% do CSD']
).astype(int)

Divisão_Supervisão_join['QTD Notas Poda'] = np.ceil(
    Divisão_Supervisão_join['QTD Supervisão'] *
    Divisão_Supervisão_join['% Notas Poda']
).astype(int)

Divisão_Supervisão_join['QTD Notas Manut'] = np.ceil(
    Divisão_Supervisão_join['QTD Supervisão'] *
    Divisão_Supervisão_join['% Notas Manut.']
).astype(int)

display(Divisão_Supervisão_join)

# Traz colunas de indicadores para tabela de notas

## Traz colunas de CI,CHI e Ocorrencias

In [ ]:
Notas

In [ ]:
#padroniza coluna de proteção final
Notas['Protecao_Final'] = np.where(
    Notas['dispositivo_protecao'].isna() | 
    (Notas['dispositivo_protecao'].astype(str).str.len() < 3),
    Notas['local_instalacao']+'9',
    Notas['dispositivo_protecao']
)

In [ ]:
Notas_Indicadores = Notas.merge(Dim_Ocorrencias, left_on='Protecao_Final', right_on='EQ_INTERRUPCAO', how='left').drop('EQ_INTERRUPCAO', axis=1)
Notas_Indicadores

In [ ]:
#subistitui valores null por 0
Notas_Indicadores = Notas_Indicadores.fillna({'QTD_Ocorrencias': 0})
Notas_Indicadores = Notas_Indicadores.fillna({'CI': 0})
Notas_Indicadores = Notas_Indicadores.fillna({'CHI': 0})
Notas_Indicadores = Notas_Indicadores.fillna({'QTD_Ocorrencias_90d': 0})

# Convertendo a coluna para inteiro
Notas_Indicadores['QTD_Ocorrencias'] = Notas_Indicadores['QTD_Ocorrencias'].astype('int64')
Notas_Indicadores['CI'] = Notas_Indicadores['CI'].astype('int64')
Notas_Indicadores['QTD_Ocorrencias_90d'] = Notas_Indicadores['QTD_Ocorrencias_90d'].astype('int64')
# Convertendo a coluna para decimal
Notas_Indicadores['CHI'] = Notas_Indicadores['CHI'].astype('float64')

Notas_Indicadores

## Cria coluna de Peso do conjunto

In [ ]:
#Versão com priorização por DEC conjunto

# Indicadores = pd.read_csv(
#     fr'{caminho_base}Base_Indicadores_Movel_ES.csv',
#     sep=";",   # tenta ; em vez de ,
#     encoding="utf-8"  # se der erro de acento, usar "latin1"
# )

# display(Indicadores)

# # Convertendo a coluna para decimal

# Indicadores['DELTA_DEC'] = (
#     Indicadores['DELTA_DEC']
#         .astype(str)
#         .str.replace(',', '.', regex=False)
#         .str.strip()
# )

# Indicadores['DELTA_DEC'] = pd.to_numeric(
#     Indicadores['DELTA_DEC'],
#     errors='coerce'
# ).fillna(0)

# # Pegando mínimo e máximo
# min_val = Indicadores['DELTA_DEC'].min()
# max_val = Indicadores['DELTA_DEC'].max()

# # Criando nova coluna com pesos
# if max_val == min_val:
#     Indicadores['Peso_Conj'] = 1
# else:
#     Indicadores['Peso_Conj'] = 1 + ((Indicadores['DELTA_DEC'] - min_val) / (max_val - min_val)) * 0.5 # Peso base ajustar se necessario 

# Indicadores

In [ ]:
Prioridade_CJ = pd.read_excel(
    fr'{caminho_base}Prioridade_Conjuntos_ENG.xlsx'
)

display(Prioridade_CJ)

In [ ]:
# Cria Peso_Conj com base na prioridade
# Regra:
# prioridade = 1 -> 1.5
# prioridade = 2 -> 1.3
# prioridade = 3 -> 1.0


# Normaliza para número (aceita texto como '1', '2', '3')
prioridade_num = pd.to_numeric(Prioridade_CJ['PRIORIDADE'], errors="coerce")

map_peso = {
    1: 2.0,
    2: 1.3,
    3: 1.0
}

Indicadores["Peso_Conj"] = prioridade_num.map(map_peso).fillna(1.0)

Indicadores


In [ ]:
Notas_Indicadores = Notas_Indicadores.merge(Indicadores[['CONJUNTO_DESC','Peso_Conj']],left_on='CJ_NOME', right_on='CONJUNTO_DESC', how='left').drop('CONJUNTO_DESC', axis=1)
Notas_Indicadores

# Cria Coluna de Cluster Final

In [ ]:
# =========================================================
# 1) TRATAMENTO E PADRONIZAÇÃO DOS DADOS
# =========================================================

# Converte colunas numéricas para float
# errors='coerce' transforma valores inválidos em NaN
Notas_Indicadores["CI"] = pd.to_numeric(
    Notas_Indicadores["CI"],
    errors="coerce"
)

Notas_Indicadores["clientes_bloco_protecao"] = pd.to_numeric(
    Notas_Indicadores["clientes_bloco_protecao"],
    errors="coerce"
).fillna(0)

Notas_Indicadores["Peso_Conj"] = pd.to_numeric(
    Notas_Indicadores["Peso_Conj"],
    errors="coerce"
).fillna(1)

# Garante que a coluna seja string
Notas_Indicadores["Protecao_Final"] = (
    Notas_Indicadores["Protecao_Final"]
    .fillna("")
    .astype(str)
)


# =========================================================
# 2) FATOR DE CLUSTER
# =========================================================

# Peso de prioridade por cluster
cluster_factor = {
    "Muito Alta": 2.0,
    "Alta": 1.7,
    "Média": 1.3,
    "Baixa": 1.0,
    "Muito Baixa": 1.0
}

# Aplica fator do cluster
Notas_Indicadores["Fator_Cluster"] = (
    Notas_Indicadores["Cluster_Priorizacao"]
    .map(cluster_factor)
    .fillna(1.0)
    .astype(float)
)


# =========================================================
# 3) FATOR DE OCORRÊNCIA 90 DIAS
# =========================================================

# Se teve ocorrência nos últimos 90 dias:
# aplica fator de aumento de prioridade
Notas_Indicadores["Fator_Ocorr90d"] = np.where(
    Notas_Indicadores["QTD_Ocorrencias_90d"].fillna(0) > 0,
    1.5,   # peso para reincidência
    1.0
).astype(float)


# =========================================================
# 4) IDENTIFICAÇÃO DE PROTEÇÕES RL E DJ
# =========================================================

# Exemplo:
# 045RL000270969
#    ^^
#    posições 4 e 5 = RL

# Em Python:
# índice começa em 0
#
# 045RL000270969
# 0123456789...
#
# Portanto:
# str[3:5] => "RL"

mask_protecao = (
    Notas_Indicadores["Protecao_Final"]
    .str[3:5]
    .isin(["RL", "DJ","CF"])
)


# =========================================================
# 5) AJUSTE DO CI
# =========================================================

# Regra:
# Se:
# - CI for nulo OU igual a 0
# - E proteção for RL ou DJ
#
# Então:
# usar clientes_bloco_protecao no lugar do CI

mask_substituir_ci = (
    (
        Notas_Indicadores["CI"].isna()
        | (Notas_Indicadores["CI"] == 0)
    )
    & mask_protecao
)

# Cria CI ajustado
ci_ajustado = (
    Notas_Indicadores["CI"]
    .mask(
        mask_substituir_ci,
        Notas_Indicadores["clientes_bloco_protecao"]
    )
    .fillna(0)
    .astype(float)
)


# =========================================================
# 6) CÁLCULO DA PRIORIDADE FINAL
# =========================================================

Notas_Indicadores["Prioridade_Final"] = (
    ci_ajustado
    * Notas_Indicadores["Fator_Cluster"]
    * Notas_Indicadores["Fator_Ocorr90d"]
    * Notas_Indicadores["Peso_Conj"]
)

# 6.1) Soma prioridade para os 34,5
Notas_Indicadores["Prioridade_Final"] = (
    Notas_Indicadores["Prioridade_Final"]
    + np.where(
        Notas_Indicadores["alimentador"]
        .astype(str)
        .str.contains("34", na=False),
        50000,
        0
    )
)

# =========================================================
# 7) RANKING DE PRIORIDADE
# =========================================================

# Quanto maior a prioridade_final,
# maior a prioridade operacional

Notas_Indicadores["Rank_Prioridade"] = (
    Notas_Indicadores["Prioridade_Final"]
    .rank(method="dense", ascending=False)
    .astype(int)
)


# =========================================================
# 8) ORDENAÇÃO FINAL
# =========================================================

Notas_Indicadores = (
    Notas_Indicadores
    .sort_values(
        by="Prioridade_Final",
        ascending=False
    )
)

# Exibe resultado
Notas_Indicadores

In [ ]:
#salva planinha no computador 
#Notas_Indicadores.to_excel(fr'{caminho_base}Notas_Rank.xlsx',index=False)

# Seleção das Notas

In [ ]:
##Faz seleeção das notas sem agruapar por equipamento não esta seno usado

# Notas_Indicadores["Notas Selecionadas"] = 0
# Notas_Indicadores["Tipo Nota Selecionada"] = pd.NA
# Notas_Indicadores["Etapa Selecionada"] = pd.NA

# faltas = []
# resumo_etapas = []

# def selecionar_em_etapas(df, base_mask, qtd_alvo, tipo_nota, csd, sup):
#     faltante = int(max(0, qtd_alvo))
#     selecionados_total = []

#     # contadores por etapa
#     etapas_info = [
#         ("1 Planejamento + Até 1 ano", "Planejamento", "Até 1 ano"),
#         ("2 Planejamento + Até 2 anos", "Planejamento", "Até 2 anos"),
#         ("3 CSD + Até 1 ano", "CSD", "Até 1 ano"),
#         ("4 CSD + Até 2 anos", "CSD", "Até 2 anos"),
#     ]
#     cont_etapas = {e[0]: 0 for e in etapas_info}

#     for nome_etapa, solicitante, classe_tempo in etapas_info:
#         if faltante <= 0:
#             break

#         mask = (
#             base_mask
#             & (df["Notas Selecionadas"] == 0)
#             & (df["Solicitante"] == solicitante)
#             & (df["Classificação Tempo"].isin([classe_tempo, "Até 2 ano"] if classe_tempo == "Até 2 anos" else [classe_tempo]))
#         )

#         cand = df.loc[mask].sort_values(
#             ["Rank_Prioridade", "Priorizacao"],
#             ascending=[True, False]
#         )

#         idx = cand.head(faltante).index
#         qtd_sel = len(idx)

#         if qtd_sel > 0:
#             df.loc[idx, "Notas Selecionadas"] = 1
#             df.loc[idx, "Tipo Nota Selecionada"] = tipo_nota
#             df.loc[idx, "Etapa Selecionada"] = nome_etapa
#             selecionados_total.extend(idx.tolist())
#             cont_etapas[nome_etapa] += qtd_sel
#             faltante -= qtd_sel

#     # resumo por CSD/Supervisão/Tipo
#     resumo_etapas.append({
#         "CSD": csd,
#         "Supervisão": sup,
#         "Tipo": tipo_nota,
#         "Meta": int(qtd_alvo),
#         "Selecionadas": int(qtd_alvo - faltante),
#         "1 Planejamento + Até 1 ano": cont_etapas["1 Planejamento + Até 1 ano"],
#         "2 Planejamento + Até 2 anos": cont_etapas["2 Planejamento + Até 2 anos"],
#         "3 CSD + Até 1 ano": cont_etapas["3 CSD + Até 1 ano"],
#         "4 CSD + Até 2 anos": cont_etapas["4 CSD + Até 2 anos"],
#         "Faltante": int(faltante),
#     })

#     return faltante, selecionados_total


# for _, r in Divisão_Supervisão_join.iterrows():
#     csd = r["CSD"]
#     sup = r["Supervisão"]

#     base_csd_sup = (
#         (Notas_Indicadores["CSD"] == csd) &
#         (Notas_Indicadores["SUPERVISAO"] == sup)
#     )

#     # PODA
#     qtd_poda = int(np.nan_to_num(r["QTD Notas Poda"], nan=0))
#     base_poda = base_csd_sup & Notas_Indicadores["descrição_conjunto"].str.contains("Poda", case=False, na=False)

#     faltante_poda, _ = selecionar_em_etapas(
#         Notas_Indicadores, base_poda, qtd_poda, "Poda", csd, sup
#     )

#     if faltante_poda > 0:
#         faltas.append({
#             "CSD": csd,
#             "Supervisão": sup,
#             "Tipo": "Poda",
#             "Quantidade Faltante": faltante_poda
#         })

#     # MANUT
#     qtd_manut = int(np.nan_to_num(r["QTD Notas Manut"], nan=0))
#     base_manut = base_csd_sup & Notas_Indicadores["descrição_conjunto"].str.contains("Manut", case=False, na=False)

#     faltante_manut, _ = selecionar_em_etapas(
#         Notas_Indicadores, base_manut, qtd_manut, "Manut", csd, sup
#     )

#     if faltante_manut > 0:
#         faltas.append({
#             "CSD": csd,
#             "Supervisão": sup,
#             "Tipo": "Manut",
#             "Quantidade Faltante": faltante_manut
#         })

# # =========================================================
# # DataFrames finais
# # =========================================================
# faltas_df = pd.DataFrame(faltas)
# resumo_etapas_df = pd.DataFrame(resumo_etapas)

# print("Resumo por CSD/Supervisão/Tipo:")
# display(resumo_etapas_df.sort_values(["CSD", "Supervisão", "Tipo"]))

# if faltas_df.empty:
#     print("Sem faltas.")
# else:
#     print("Faltas encontradas:")
#     display(faltas_df.sort_values(["CSD", "Supervisão", "Tipo"]))


In [ ]:
# Faz seleção das notas pelo racking ja definido e estapas de filtros
#Agrupar notas de mesmo bloco de manutenção do planejamento
Notas_Indicadores["Notas Selecionadas"] = 0
Notas_Indicadores["Tipo Nota Selecionada"] = pd.NA
Notas_Indicadores["Etapa Selecionada"] = pd.NA

faltas = []
resumo_etapas = []

def selecionar_em_etapas(df, base_mask, qtd_alvo, tipo_nota, csd, sup, agrupar_por_protecao=False):
    faltante = int(max(0, qtd_alvo))
    selecionados_total = []

    etapas_info = [
        ("1 Planejamento + Até 1 ano", "Planejamento", "Até 1 ano"),
        ("2 Planejamento + Até 2 anos", "Planejamento", "Até 2 anos"),
        ("3 CSD + Até 1 ano", "CSD", "Até 1 ano"),
        ("4 CSD + Até 2 anos", "CSD", "Até 2 anos"),
    ]
    cont_etapas = {e[0]: 0 for e in etapas_info}

    for nome_etapa, solicitante, classe_tempo in etapas_info:
        if faltante <= 0:
            break

        classes_tempo = [classe_tempo, "Até 2 ano"] if classe_tempo == "Até 2 anos" else [classe_tempo]

        mask = (
            base_mask
            & (df["Notas Selecionadas"] == 0)
            & (df["Solicitante"] == solicitante)
            & (df["Classificação Tempo"].isin(classes_tempo))
        )

        cand = df.loc[mask].sort_values(
            ["Rank_Prioridade", "Priorizacao"],
            ascending=[True, False]
        )

        if cand.empty:
            continue

        if not agrupar_por_protecao:
            idx_sel = cand.head(faltante).index.tolist()
        else:
            # Manut: percorre equipamentos por ordem de prioridade
            # e seleciona o bloco do mesmo Protecao_Final antes de ir ao próximo.
            cand = cand.copy()
            cand["_ordem"] = np.arange(len(cand))
            cand["_prot_key"] = np.where(
                cand["Protecao_Final"].notna(),
                cand["Protecao_Final"].astype(str),
                "__SEM_PROTECAO__" + cand.index.astype(str)
            )

            grupos = (
                cand.groupby("_prot_key", sort=False)
                .agg(
                    primeira_ordem=("_ordem", "min"),
                    idx=("Rank_Prioridade", lambda s: s.index.tolist())
                )
                .sort_values("primeira_ordem")
            )

            idx_sel = []
            for _, g in grupos.iterrows():
                if faltante <= 0:
                    break

                idx_grupo = g["idx"]
                if len(idx_grupo) <= faltante:
                    idx_add = idx_grupo
                else:
                    idx_add = idx_grupo[:faltante]  # até preencher

                idx_sel.extend(idx_add)
                faltante -= len(idx_add)

        qtd_sel = len(idx_sel)
        if qtd_sel > 0:
            df.loc[idx_sel, "Notas Selecionadas"] = 1
            df.loc[idx_sel, "Tipo Nota Selecionada"] = tipo_nota
            df.loc[idx_sel, "Etapa Selecionada"] = nome_etapa
            selecionados_total.extend(idx_sel)
            cont_etapas[nome_etapa] += qtd_sel

            # No fluxo sem agrupamento, faltante ainda não foi abatido
            if not agrupar_por_protecao:
                faltante -= qtd_sel

    resumo_etapas.append({
        "CSD": csd,
        "Supervisão": sup,
        "Tipo": tipo_nota,
        "Meta": int(qtd_alvo),
        "Selecionadas": int(qtd_alvo - faltante),
        "1 Planejamento + Até 1 ano": cont_etapas["1 Planejamento + Até 1 ano"],
        "2 Planejamento + Até 2 anos": cont_etapas["2 Planejamento + Até 2 anos"],
        "3 CSD + Até 1 ano": cont_etapas["3 CSD + Até 1 ano"],
        "4 CSD + Até 2 anos": cont_etapas["4 CSD + Até 2 anos"],
        "Faltante": int(faltante),
    })

    return faltante, selecionados_total


for _, r in Divisão_Supervisão_join.iterrows():
    csd = r["CSD"]
    sup = r["Supervisão"]

    base_csd_sup = (
        (Notas_Indicadores["CSD"] == csd) &
        (Notas_Indicadores["SUPERVISAO"] == sup)
    )

    # PODA
    qtd_poda = int(np.nan_to_num(r["QTD Notas Poda"], nan=0))
    base_poda = base_csd_sup & Notas_Indicadores["descrição_conjunto"].str.contains("Poda", case=False, na=False)

    faltante_poda, _ = selecionar_em_etapas(
        Notas_Indicadores, base_poda, qtd_poda, "Poda", csd, sup, agrupar_por_protecao=False
    )

    if faltante_poda > 0:
        faltas.append({
            "CSD": csd,
            "Supervisão": sup,
            "Tipo": "Poda",
            "Quantidade Faltante": faltante_poda
        })

    # MANUT (agrupando por Protecao_Final)
    qtd_manut = int(np.nan_to_num(r["QTD Notas Manut"], nan=0))
    base_manut = base_csd_sup & Notas_Indicadores["descrição_conjunto"].str.contains("Manut", case=False, na=False)

    faltante_manut, _ = selecionar_em_etapas(
        Notas_Indicadores, base_manut, qtd_manut, "Manut", csd, sup, agrupar_por_protecao=True
    )

    if faltante_manut > 0:
        faltas.append({
            "CSD": csd,
            "Supervisão": sup,
            "Tipo": "Manut",
            "Quantidade Faltante": faltante_manut
        })

# =========================================================
# DataFrames finais
# =========================================================
faltas_df = pd.DataFrame(faltas)
resumo_etapas_df = pd.DataFrame(resumo_etapas)

print("Resumo por CSD/Supervisão/Tipo:")
display(resumo_etapas_df.sort_values(["CSD", "Supervisão", "Tipo"]))

if faltas_df.empty:
    print("Sem faltas.")
else:
    print("Faltas encontradas:")
    display(faltas_df.sort_values(["CSD", "Supervisão", "Tipo"]))


In [ ]:
Notas_Indicadores["Tipo dispositivo"] = Notas_Indicadores["Protecao_Final"].str[3:5]

# Salva tabelas finais

In [ ]:
Notas_Indicadores

In [ ]:
Notas_Indicadores.to_excel(fr'{caminho_base}Notas_Selecionadas.xlsx',index=False)

In [ ]:
with pd.ExcelWriter(fr"{caminho_base}Resumo_Selecao_Notas.xlsx", engine="openpyxl") as writer:
    resumo_etapas_df.to_excel(writer, sheet_name="resumo_etapas", index=False)
    faltas_df.to_excel(writer, sheet_name="faltas", index=False)